In [ ]:
import pandas as pd
import json
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report



with open('/home/cecilia/Documentos/PIBIC/Fase2/Dados_SVM-BERT/train_bio.json', 'r', encoding='utf-8') as f:
    df_train = pd.DataFrame(json.load(f))



with open('/home/cecilia/Documentos/PIBIC/Fase2/Dados_SVM-BERT/val_bio.json', 'r', encoding='utf-8') as f:
    df_val = pd.DataFrame(json.load(f))


with open('/home/cecilia/Documentos/PIBIC/Fase2/Dados_SVM-BERT/test_bio.json', 'r', encoding='utf-8') as f:
    df_test = pd.DataFrame(json.load(f))


In [8]:
def montar_tags(linha):
    rotulos = []

    categoria = linha['categoria']

    for tag in linha['tags']:
        if tag == 'O':
            tag = 'O'
            rotulos.append(tag)

        elif tag == 'B-ASP':
            tag = f'B-ASP|{categoria}'
            rotulos.append(tag)

        elif tag == 'I-ASP':
            tag = f'I-ASP|{categoria}'
            rotulos.append(tag)

    return rotulos

In [9]:
df_train['rotulos'] = df_train.apply(montar_tags, axis=1)
df_val['rotulos'] = df_val.apply(montar_tags, axis=1)
df_test['rotulos'] = df_test.apply(montar_tags, axis=1)



In [10]:
X_train_tokens = [str(token) for sublist in df_train['tokens'] for token in sublist]
y_train_tags = [tag for sublist in df_train['rotulos'] for tag in sublist]

In [ ]:
print("Total de tokens treino:", len(X_train_tokens))
print("Número de classes únicas:", len(set(y_train_tags)))

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

pipeline_svm = make_pipeline(
    TfidfVectorizer(analyzer='word', token_pattern=r'\S+'),
    SGDClassifier(loss='hinge', class_weight='balanced', n_jobs=-1, random_state=42, max_iter=1000)
)

print("Treinando o modelo LinearSVC...")
pipeline_svm.fit(X_train_tokens, y_train_tags)
print("Modelo treinado com sucesso!")

In [15]:
X_val_tokens = [str(token) for sublist in df_test['tokens'] for token in sublist]
y_val_tags = [tag for sublist in df_test['rotulos'] for tag in sublist]

y_pred_tags = pipeline_svm.predict(X_val_tokens)

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

precisao = precision_score(y_val_tags, y_pred_tags, average='weighted', zero_division=0)
recall = recall_score(y_val_tags, y_pred_tags, average='weighted', zero_division=0)
f1 = f1_score(y_val_tags, y_pred_tags, average='weighted', zero_division=0)
acuracia = accuracy_score(y_val_tags, y_pred_tags)

print(f"Precisão: {precisao:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")
print(f"Acurácia: {acuracia:.4f}")